# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the FAIR² mass spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL provided by Sen Science FAIR².

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Display main metadata about the dataset
print(f"Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Version: {dataset.metadata.version}")
print(f"Identifier: {dataset.metadata.identifier}")
print(f"Published: {dataset.metadata.datePublished}")

## 2. Data Overview

Review available record sets, fields, and their `@id`s.

In [ ]:
# List available record sets (@id) and their fields
print("Record sets in dataset:")
if not dataset.metadata.recordSet:
    print("No record sets defined in metadata; inferring from records.")
    # Try to infer possible record sets from the implementation
    # The mlcroissant API may expose this via dataset.record_sets
    # For Croissant 1.0, sometimes the record sets are named in the 'record_sets' dict
    # Let's print them dynamically
    if hasattr(dataset, 'record_sets'):
        for rs_id in dataset.record_sets:
            print(f"- {rs_id}")
    else:
        print("Unable to list record sets. Please refer to the Croissant file or documentation.")
else:
    for rs in dataset.metadata.recordSet:
        print(f"- @id: {rs['@id']}, name: {rs.get('name', '(no name)')}")
        if 'field' in rs:
            fields = rs['field']
            if isinstance(fields, dict):
                fields = [fields]
            for fld in fields:
                print(f"    * field @id: {fld['@id']}, name: {fld.get('name', '(no name)')}")

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Identify the available record sets
record_sets_ids = []
if hasattr(dataset, 'record_sets'):
    record_sets_ids = list(dataset.record_sets.keys())

# If only one record set guess, use it
if not record_sets_ids:
    # Try default record set name used by mlcroissant, e.g., if only one is present
    record_sets_ids = ['default']

# Preview content of available record sets
dataframes = {}
for rs_id in record_sets_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f'Loaded {len(df)} records for record set "{rs_id}"')
        print('Columns:', df.columns.tolist())
        display(df.head())
    except Exception as e:
        print(f"Error loading record set {rs_id}: {e}")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Select which record set to use (if multiple are present, update this accordingly)
record_set_id = record_sets_ids[0]  # Use the first discovered record set
df = dataframes[record_set_id]

# List candidate numeric fields for analysis
numeric_candidate_fields = df.select_dtypes(include='number').columns.tolist()
print('Numeric fields in the DataFrame:', numeric_candidate_fields)

# If available, select a numeric field (e.g., abundance, MW, etc.)
if numeric_candidate_fields:
    numeric_field = numeric_candidate_fields[0]
else:
    raise ValueError('No numeric fields available for EDA.')

# Set a threshold (customize as needed)
threshold = df[numeric_field].mean() if len(df) > 0 else 0

# Filter DataFrame by the chosen threshold
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold}:")
display(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Attempt a groupby using a suitable categorical field, if present
categorical_fields = df.select_dtypes(include=['object']).columns.tolist()
group_field = None
for col in categorical_fields:
    if col.lower() not in ['id', 'accession', 'name', 'description'] and df[col].nunique() < min(30, len(df)//2):
        group_field = col
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().sort_values(by=numeric_field, ascending=False)
    print(f"Grouped data by {group_field}, showing mean {numeric_field}:")
    display(grouped_df.head())
else:
    print("No suitable categorical field found for grouping.")

## 5. Visualization

Visualize distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field before and after normalization
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f'Histogram of {numeric_field}')
plt.xlabel(numeric_field)

plt.subplot(1,2,2)
sns.histplot(filtered_df[f'{numeric_field}_normalized'].dropna(), bins=30, kde=True, color='orange')
plt.title(f'Histogram of Normalized {numeric_field} (filtered)')
plt.xlabel(f"{numeric_field}_normalized")
plt.tight_layout()
plt.show()

# If grouping was possible, visualize the mean value per category
if 'grouped_df' in locals() and group_field:
    plt.figure(figsize=(10,6))
    sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
    plt.title(f'Mean {numeric_field} by {group_field}')
    plt.xticks(rotation=45, ha='right')
    plt.show()

## 6. Conclusion

In this notebook, we used `mlcroissant` to load and explore the FAIR² dataset covering mass spectrometry protein analyses in human mast cell extracellular vesicles.

- We identified the main record sets via their `@id` values.
- We loaded the data into pandas DataFrames for flexible exploration and analysis.
- We performed example exploratory data analysis including threshold filtering, normalization, and grouping by categorical attributes.
- We visualized distributions of key numeric fields and their normalized versions.

This workflow demonstrates the power of Croissant metadata and the `mlcroissant` Python library for FAIR biomedical data exploration. For deeper research, repeat these steps with different fields, filters, or visualizations, referencing all field and record set names by their `@id` for maximum reproducibility.